### Section 2 - Indexing, selection & Copy-on-write

In [61]:
import tempfile
import pathlib
import numpy as np
import pandas as pd

assert pd.__version__.startswith("3."), pd.__version__

def make_books() -> pd.DataFrame:
    return pd.DataFrame({
        "title": ["Dune", "Neuromancer", "Foundation", "Hyperion",
                  "The Left Hand of Darkness", "Snow Crash"],
        "author": ["Herbert", "Gibson", "Asimov", "Simmons", "Le Guin", "Stephenson"],
        "year": [1965, 1984, 1951, 1989, 1969, 1992],
        "pages": [412, 271, 255, 482, 304, 480],
        "price": [18.99, 14.50, 12.00, 17.25, 15.75, 16.40],
    })

In [62]:
# Exercise 2.1
b = make_books()
b.index = [10, 20, 30, 40, 50, 60]

print(b.loc[20:40, "title"].tolist())   # ["Neuromancer", "Foundation", "Hyperion"] 
print(b.iloc[1:3]["title"].tolist())    # ["Neuromancer", "Foundation"] 
print(type(b.loc[20]).__name__)         # Series
print(b.iloc[-1]["title"])              # Snow Crash

['Neuromancer', 'Foundation', 'Hyperion']
['Neuromancer', 'Foundation']
Series
Snow Crash


In [63]:
picked = b.loc[[30, 50], ["title", "price"]]
first_two = b.iloc[:2, :2]

assert picked.shape == (2, 2)
assert picked["title"].tolist() == ["Foundation", "The Left Hand of Darkness"]
assert first_two.to_dict() == {"title": {10: "Dune", 20: "Neuromancer"},
                               "author": {10: "Herbert", 20: "Gibson"}}
b

,title,author,year,pages,price
10,Dune,Herbert,1965,412,18.99
20,Neuromancer,Gibson,1984,271,14.50
30,Foundation,Asimov,1951,255,12.00
40,Hyperion,Simmons,1989,482,17.25
50,The Left Hand of Darkness,Le Guin,1969,304,15.75
60,Snow Crash,Stephenson,1992,480,16.40


In [64]:
# Exercise 2.2
books = make_books()

old_and_pricey = books[(books["year"] < 1980) & (books["price"] > 15)]
cyber = books[books["author"].isin(["Gibson", "Stephenson"])]
mid = books[books["pages"].between(300, 450)]
not_asimov = books[~(books["author"] == "Asimov")]

assert old_and_pricey["title"].tolist() == ["Dune", "The Left Hand of Darkness"]
assert cyber["title"].tolist() == ["Neuromancer", "Snow Crash"]
assert mid["title"].tolist() == ["Dune", "The Left Hand of Darkness"]
assert len(not_asimov) == 5
# when i run the line below it raises error, because it is doing AND operator first: books["year"] < (1980 & books["price"]) > 15
# books[books["year"] < 1980 & books["price"] > 15]
# in python, '&' has higher precedence than comparison operators like '<' and '>'
# it should look like this: books[(books["year"] < 1980) & (books["price"] > 15)]

In [65]:
# Exercise 2.3
# A
books = make_books()
cheap = books[books["price"] < 15]
cheap.loc[:, "price"] = 0.0
print(books["price"].tolist(), cheap["price"].tolist()) # [18.99, 14.5, 12.0, 17.25, 15.75, 16.4], [0.0, 0.0]

books = make_books()
books["price"][books["year"] < 1960] = 99.0   # books doesn't change because this assigning creates copy on first step, then assign value to this copy, pandas gives ChainedAssignmentError due to Copy-on-Write rule
print(books["price"].tolist())

# B
books = make_books()
books.loc[books["year"] < 1970, "price"] *= 0.9
assert np.allclose(books["price"], [17.091, 14.50, 10.80, 17.25, 14.175, 16.40])


[18.99, 14.5, 12.0, 17.25, 15.75, 16.4] [0.0, 0.0]
[18.99, 14.5, 12.0, 17.25, 15.75, 16.4]


C:\Users\Kacba\AppData\Local\Temp\ipykernel_27860\3718259372.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  books["price"][books["year"] < 1960] = 99.0   # books doesn't change because this assigning creates copy on first step, then assign value to this copy, pandas gives ChainedAssignmentError due to Copy-on-Write rule


In [66]:
# Exercise 2.4
arr = np.array([1.0, 2.0, 3.0])
view = arr[:2]
view[0] = 100.0
print(arr)                  # [100.0, 2.0, 3.0]

s = pd.Series([1.0, 2.0, 3.0])
sub = s[:2]
sub.iloc[0] = 100.0
print(s.tolist())           # [1.0, 2.0, 3.0]

vals = s.to_numpy()
# vals[0] = 100.0           # assigning value to numpy array which is view only - gives ValueError: assignment destination is read-only

doubled = s.to_numpy(copy=True)
doubled *= 2
assert doubled.tolist() == [2.0, 4.0, 6.0]
assert s.tolist() == [1.0, 2.0, 3.0]
# NumPy: low-level, optimized number library → users want speed ,
#        and the rule is simple (basic slice = always a view), so views are safe enough.
# pandas: high-level analysis tool → users select subsets to analyze , not to edit the original;
#         old pandas mixed views and copies unpredictably → silent changes to the original data.
# Copy-on-Write keeps this cheap: data is shared until modified .

[100.   2.   3.]
[1.0, 2.0, 3.0]
